In [1]:
from google.colab import files
uploaded=files.upload()

Saving accounts.csv to accounts.csv
Saving branches.csv to branches.csv
Saving customer_profiles.json to customer_profiles.json
Saving customers.csv to customers.csv
Saving logs.txt to logs.txt


In [6]:
from google.colab import files
uploaded=files.upload()

Saving transactions.csv to transactions.csv


In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("BankingAnalytics").getOrCreate()
customers=spark.read.csv("customers.csv",header=True,inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)

In [14]:
customers.show()
accounts.show()
transactions.show()
branches.show()
customers.printSchema()
transactions.printSchema()
customers.count()
accounts.count()
transactions.count()
transactions.show(5)

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+

+----------+-----------+---------------

In [15]:
customers.select("name","city","account_type").show()
accounts.select("account_id","balance").show()

accounts.withColumnRenamed("balance","current_balance").show()
transactions.withColumnRenamed("txn_type","transaction_type").show()

customers.filter("city = 'Hyderabad'").show()
customers.filter("age > 30").show()
customers.filter("account_type = 'Savings'").show()

accounts.filter("balance > 100000").show()
transactions.filter("txn_type = 'Credit'").show()
transactions.filter("amount > 20000").show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+

+----------+-----------+-----------------+---------------+
|account_id|customer_id|           branch|current_balance|
+----------+-----------+--------

In [24]:
customers.orderBy("age").show()
customers.orderBy("age",ascending=False).show()
accounts.orderBy("balance",ascending=False).show()
accounts.orderBy("balance",ascending=False).show(5)
accounts.orderBy("balance").show(5)
transactions.orderBy("amount",ascending=False).show()
transactions.orderBy("amount",ascending=False).show(3)
transactions.orderBy("txn_date").show()
customers.orderBy("city","age").show()
transactions.orderBy("txn_date",ascending=False).show(5)

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+

+-----------+------+---------+---+-----

In [25]:
accounts.agg(sum("balance")).show()
accounts.agg(avg("balance")).show()
accounts.agg(max("balance")).show()
accounts.agg(min("balance")).show()
customers.groupBy("city").count().show()
customers.groupBy("account_type").count().show()
transactions.groupBy("txn_type").count().show()
transactions.filter(col("txn_type")=="Credit").agg(sum("amount")).show()
transactions.filter(col("txn_type")=="Debit").agg(sum("amount")).show()
transactions.agg(avg("amount")).show()

+------------+
|sum(balance)|
+------------+
|     1055000|
+------------+

+-----------------+
|     avg(balance)|
+-----------------+
|87916.66666666667|
+-----------------+

+------------+
|max(balance)|
+------------+
|      200000|
+------------+

+------------+
|min(balance)|
+------------+
|       30000|
+------------+

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|   10|
|   Debit|   10|
+--------+-----+

+-----------+
|sum(amount)|
+-----------+
|     279000|
+-----------+

+-----------+
|sum(amount)|
+-----------+
|     132000|
+-----------+

+-----------+
|avg(amount)|
+-----------+
|    20550.0|
+-----------+



In [26]:
accounts.groupBy("branch").agg(sum("balance")).show()
accounts.groupBy("branch").agg(avg("balance")).show()
accounts.groupBy("branch").count().show()
transactions.groupBy("account_id").agg(sum("amount")).show()
transactions.groupBy("account_id").count().show()
transactions.groupBy("txn_type").agg(sum("amount")).show()
transactions.groupBy("txn_date").count().show()
customers.groupBy("city").agg(avg("age")).show()
customers.join(accounts,"customer_id").groupBy("account_type").agg(sum("balance")).show()
customers.join(accounts,"customer_id").groupBy("city","account_type").count().show()

+-----------------+------------+
|           branch|sum(balance)|
+-----------------+------------+
|      Delhi North|      205000|
|   Hyderabad Main|      285000|
|        Pune East|       30000|
|      Mumbai West|      245000|
|    Chennai South|       60000|
|Bangalore Central|      230000|
+-----------------+------------+

+-----------------+-----------------+
|           branch|     avg(balance)|
+-----------------+-----------------+
|      Delhi North|68333.33333333333|
|   Hyderabad Main|          95000.0|
|        Pune East|          30000.0|
|      Mumbai West|         122500.0|
|    Chennai South|          60000.0|
|Bangalore Central|         115000.0|
+-----------------+-----------------+

+-----------------+-----+
|           branch|count|
+-----------------+-----+
|      Delhi North|    3|
|   Hyderabad Main|    3|
|        Pune East|    1|
|      Mumbai West|    2|
|    Chennai South|    1|
|Bangalore Central|    2|
+-----------------+-----+

+----------+-----------+
|a

In [27]:
cust_acc = customers.join(accounts,"customer_id")
cust_acc.show()
cust_acc.select("name","city","branch","balance").show()
acc_txn = accounts.join(transactions,"account_id")
acc_txn.select("account_id","txn_type","amount","balance").show()
acc_branch = accounts.join(branches,"branch")
acc_branch.select("branch","region","manager","balance").show()
full = customers.join(accounts,"customer_id").join(transactions,"account_id")
full.select("name","city","txn_type","amount","txn_date").show()

full2 = customers.join(accounts,"customer_id").join(branches,"branch").join(transactions,"account_id")

full2.groupBy("name").agg(sum("amount")).show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|      1001|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      1003|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|
|          8|  Neha|    Delhi|

In [ ]:
accounts.withColumn("balance_in_lakhs", expr("balance/100000")).show()
accounts.withColumn("bank_name", lit("BotCampus Bank")).show()
accounts.withColumn("annual_service_fee", expr("balance * 0.01")).show()
accounts.withColumn("net_balance", expr("balance - (balance * 0.01)")).show()
accounts.withColumn("is_high_balance", expr("balance > 100000")).show()
transactions.withColumn("txn_amount_in_k", expr("amount/1000")).show()
customers.withColumn("country", lit("India")).show()
customers.withColumn("customer_label", expr("concat(name,' - ',city)")).show()
accounts.join(branches,"branch").withColumn("branch_label", expr("concat(branch,' - ',region)")).show()
transactions.withColumn("risk_flag", expr("amount > 40000")).show()

In [28]:
accounts.withColumn("balance_category",when(accounts.balance >= 100000,"High").when(accounts.balance >= 50000,"Medium").otherwise("Low")).show()
customers.withColumn("age_group", when(customers.age < 30,"Young").when(customers.age < 40,"Adult").otherwise("Senior")).show()
transactions.withColumn("txn_category", when(transactions.amount >= 30000,"Large") .when(transactions.amount >= 10000,"Medium").otherwise("Small")).show()
branches.withColumn("priority", when(branches.region == "South","High Priority").when(branches.region == "North","Medium Priority").otherwise("Watch")).show()
customers.withColumn("type_label", when(customers.account_type == "Savings","Retail").otherwise("Business")).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_category|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|          Medium|
|      1002|          2|Bangalore Central| 120000|            High|
|      1003|          3|      Mumbai West|  45000|             Low|
|      1004|          4|      Delhi North|  95000|          Medium|
|      1005|          5|    Chennai South|  60000|          Medium|
|      1006|          6|   Hyderabad Main| 150000|            High|
|      1007|          7|        Pune East|  30000|             Low|
|      1008|          8|      Delhi North|  70000|          Medium|
|      1009|          9|Bangalore Central| 110000|            High|
|      1010|         10|      Mumbai West| 200000|            High|
|      1011|         11|   Hyderabad Main|  50000|          Medium|
|      1012|         12|      Delhi North|  4000

In [29]:
customers = customers.withColumn("signup_date", to_date("signup_date"))
transactions = transactions.withColumn("txn_date", to_date("txn_date"))
customers.withColumn("year", year("signup_date")).show()
customers.withColumn("month", month("signup_date")).show()
transactions.withColumn("month", month("txn_date")).show()
transactions.groupBy("txn_date").count().show()
transactions.groupBy(month("txn_date")).count().show()
customers.filter("signup_date > '2023-06-01'").show()
customers.withColumn("days_since_signup",datediff(current_date(),"signup_date")).show()
transactions.withColumn("days_since_txn",datediff(current_date(),"txn_date")).show()

+-----------+------+---------+---+------------+-----------+----+
|customer_id|  name|     city|age|account_type|signup_date|year|
+-----------+------+---------+---+------------+-----------+----+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|2023|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|2023|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|2023|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|2023|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|2023|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|2023|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|2023|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|2023|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|2023|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|2023|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|2023|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|2023|
+-----------+------+-----

In [32]:
w_city = Window.partitionBy("city").orderBy(expr("balance DESC"))
customers.join(accounts,"customer_id").withColumn("rank", rank().over(w_city)).show()
customers.join(accounts,"customer_id").withColumn("row_num", row_number().over(w_city)).filter("row_num = 1").show()
w_txn = Window.partitionBy("txn_type").orderBy(expr("amount DESC"))
transactions.withColumn("dense_rank", dense_rank().over(w_txn)).show()
w_acc = Window.partitionBy("account_id").orderBy(expr("amount DESC"))
transactions.withColumn("rank", rank().over(w_acc)).filter("rank <= 2").show()
transactions.withColumn("running_total",sum("amount").over(Window.partitionBy("account_id").orderBy("txn_date"))).show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|rank|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|      1009|Bangalore Central| 110000|   1|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|   2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|   1|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|      1012|      Delhi North|  40000|   1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      1008|      Delhi North|  70000|   2|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|   3|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|      1011|   Hyderabad